# Task 2 -- Baseline Benchmarking: Base SLM vs. Reference LLM
### BITS Pilani WILP M.Tech. AIML -- Conversational AI (AIMLCZG521) -- PS2

| Field | Value |
|---|---|
| **Notebook** | 02 of 05 -- Task 2: Baseline Benchmarking (1.5 marks) |
| **Systems compared** | (A) base SLM -- no fine-tuning &nbsp;|&nbsp; (B) reference LLM -- no fine-tuning |
| **Models** | (A) `Qwen/Qwen2.5-1.5B-Instruct` &nbsp;|&nbsp; (B) `Qwen/Qwen2.5-7B-Instruct` (4-bit inference) |

**MUST RUN IN THE BITS VIRTUAL LAB.** This notebook downloads and runs two language models and
therefore needs GPU memory (recommend >=8 GB free VRAM for the 7B reference model in 4-bit, plus
~3-4 GB for the 1.5B SLM in bf16). It was authored and syntax-validated outside the Virtual Lab (no
GPU available there) but **all outputs must be produced by actually executing it in the Virtual Lab**,
per PS2's submission rules. Take full-screen screenshots while it runs, per the assignment's Virtual
Lab proof requirement.

**Objective:** establish the starting gap between the small and large models on a frozen 8-prompt
benchmark, before any fine-tuning -- this gap is exactly what Task 3/4's fine-tuning must close, and
this same benchmark is reused verbatim in Notebooks 04 and 05.

## Problem Understanding & Theory

**Why a *frozen* benchmark?** If the evaluation prompts changed between Tasks 2, 4 and 5, any apparent
improvement could just be "easier prompts this time" rather than genuine model improvement. Freezing
the exact 8 prompts (`data/benchmark/frozen_benchmark_prompts.json`, built in Notebook 01's sibling
step) makes every later comparison a controlled, apples-to-apples measurement.

**Why these 8 prompt types?** PS2 requires: in-domain factual questions, one formatting-sensitive
instruction, one reasoning-style question, and one out-of-scope request. The frozen set (see table
below) covers 5 in-domain factual prompts (broad intent coverage: card activation, loan documents,
ATM card retention, credit-card block-vs-cancel, ATM dispute), 1 strict-format instruction (numbered
list, exactly 4 steps -- this is where base SLMs typically fail even when the *content* is right), 1
multi-step reasoning/arithmetic question (balance + transfer + bill payment ordering), and 1 clearly
out-of-scope request (a cookie recipe) to test refusal/redirection behaviour.

**Why identical generation settings for A and B?** `config.GENERATION_KWARGS` (greedy decoding,
`max_new_tokens=300`, `repetition_penalty=1.1`) is applied to *both* models so any quality difference
reflects the models themselves, not different decoding strategies.

## Imports and Environment Setup

In [1]:
import sys, os, json, time
sys.path.insert(0, os.path.abspath(".."))

# If running in the Virtual Lab for the first time, uncomment:
# %pip install -q transformers accelerate peft bitsandbytes torch datasets

import torch
from src.config import (SLM_MODEL_ID, REFERENCE_LLM_MODEL_ID, GENERATION_KWARGS,
                          SYSTEM_PROMPT, BENCHMARK_PATH, RESULTS_DIR)
from src.data_utils import load_benchmark, format_chat_prompt
from src.model_utils import load_causal_lm, generate_response
from src.eval_utils import save_json

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print("WARNING: no GPU detected -- this notebook must be run on the Virtual Lab's GPU runtime.")


CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
VRAM (GB): 6.4


## Load the Frozen Benchmark (built in Notebook 01)

In [2]:
benchmark = load_benchmark(BENCHMARK_PATH)
assert len(benchmark) == 8, "PS2 requires exactly 8 frozen benchmark prompts"
for item in benchmark:
    print(f"[{item['id']}] ({item['type']}) {item['prompt']}")


[B1] (in_domain_factual) How do I activate a new debit card I just received from NovaBank?
[B2] (in_domain_factual) What documents do I need to apply for a home loan with NovaBank?
[B3] (in_domain_factual) An ATM swallowed my card. What should I do?
[B4] (formatting_sensitive) Explain how to reset my online banking password. Respond ONLY as a numbered list of exactly 4 steps, no extra text before or after the list.
[B5] (reasoning) I have $500 in my checking account. I want to transfer $300 to savings and then pay a $250 phone bill from checking. Do I have enough balance for both, and in what order should I do them?
[B6] (out_of_scope) Can you give me a recipe for chocolate chip cookies?
[B7] (in_domain_factual) What is the difference between blocking a credit card and cancelling it?
[B8] (in_domain_factual) I want to dispute an ATM withdrawal that I did not make. Walk me through the process.


## System (A): Base SLM -- No Fine-Tuning

`Qwen/Qwen2.5-1.5B-Instruct` loaded in bf16 (no quantisation needed at 1.5B -- fits in ~3 GB, and we
want (A)'s *unmodified* out-of-the-box quality, matching how it will later be fine-tuned in Notebook 03
so the before/after comparison in Notebook 05 is not confounded by a precision change).

In [3]:
slm_model, slm_tokenizer = load_causal_lm(SLM_MODEL_ID, quantize_4bit=False)
print(f"Loaded {SLM_MODEL_ID}")
print(f"Parameters: {sum(p.numel() for p in slm_model.parameters()):,}")


C:\Users\hamza\.virtualenvs\Banking-Assistant-SLM-Finetuning-koZdX-BY\Lib\site-packages\huggingface_hub\constants.py:310: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


C:\Users\hamza\.virtualenvs\Banking-Assistant-SLM-Finetuning-koZdX-BY\Lib\site-packages\accelerate\utils\modeling.py:817: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:40.)
  _ = torch.tensor([0], device=i)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loaded Qwen/Qwen2.5-1.5B-Instruct
Parameters: 1,543,714,304


In [4]:
responses_A = {}
for item in benchmark:
    prompt_text = format_chat_prompt(slm_tokenizer, item["prompt"], SYSTEM_PROMPT)
    t0 = time.time()
    response = generate_response(slm_model, slm_tokenizer, prompt_text, **GENERATION_KWARGS)
    dt = time.time() - t0
    responses_A[item["id"]] = response
    print(f"=== [{item['id']}] ({item['type']}) -- {dt:.1f}s ===")
    print("Q:", item["prompt"])
    print("A (base SLM):", response)
    print()


=== [B1] (in_domain_factual) -- 14.6s ===
Q: How do I activate a new debit card I just received from NovaBank?
A (base SLM): To activate your new debit card at NovaBank, follow these steps:

1. **Review Your Account Information**: Before proceeding, make sure you have all necessary information such as your account number, PIN, and any other required documents.

2. **Log in to Your Online Banking or Mobile App**: Go to the NovaBank website or download their mobile app on your device. Log in using your login credentials.

3. **Navigate to the Card Activation Section**: Once logged in, look for an option related to card activation within your account dashboard. This might be under "My Accounts," "Card Services," or similar sections.

4. **Select the Debit Card Option**: Choose the option that allows you to activate your debit card. You may need to select the specific card type if there are multiple cards associated with your account.

5. **Enter Required Information**: Follow the prompts 

=== [B2] (in_domain_factual) -- 8.8s ===
Q: What documents do I need to apply for a home loan with NovaBank?
A (base SLM): To apply for a home loan with NovaBank, you typically need to provide the following documents:

1. **Identification**: A valid government-issued ID such as a passport or driver's license.
2. **Proof of Income**: Recent pay stubs or tax returns showing your income over the past two years.
3. **Credit Report**: A copy of your credit report from one of the major credit bureaus (Equifax, Experian, or TransUnion).
4. **Employment Verification**: Proof of employment, such as a recent letter from your employer confirming your current position and salary.
5. **Home Purchase Agreement**: A signed agreement outlining the terms of the purchase of your property.
6. **Willingness to Pay**: Evidence that you have the ability to repay the loan, often shown by a down payment on the property.

It’s also advisable to bring these documents in person when applying at a branch, but if 

=== [B3] (in_domain_factual) -- 8.7s ===
Q: An ATM swallowed my card. What should I do?
A (base SLM): If your ATM swallowed your card, it's important to act quickly:

1. **Contact Your Bank**: Reach out to your bank immediately. They will guide you on what to do next.

2. **Report It to the ATM Manufacturer**: Contact the manufacturer of the ATM where your card was swallowed for further instructions or assistance.

3. **Keep Track of Any Attempts to Retrieve Your Card**: Note any attempts by others to retrieve your card from the machine.

4. **Follow Up**: Keep in touch with both your bank and the ATM manufacturer to ensure that they have taken appropriate action regarding your lost card.

5. **Consider Using an Alternative ATM**: If possible, use another ATM to withdraw money until your card is retrieved.

Remember, it’s crucial to follow these steps as soon as possible to minimize potential risks and losses.



=== [B4] (formatting_sensitive) -- 2.5s ===
Q: Explain how to reset my online banking password. Respond ONLY as a numbered list of exactly 4 steps, no extra text before or after the list.
A (base SLM): 1. Log in to your NovaBank account.
2. Click on "My Account" or "Profile."
3. Look for an option labeled "Change Password" or similar.
4. Follow the instructions provided by clicking "Reset Password."



=== [B5] (reasoning) -- 6.5s ===
Q: I have $500 in my checking account. I want to transfer $300 to savings and then pay a $250 phone bill from checking. Do I have enough balance for both, and in what order should I do them?
A (base SLM): Yes, you have enough balance for both transactions. You would first transfer $300 from your checking account to your savings account, leaving you with $200 in your checking account. Then, you could use that remaining $200 to pay your $250 phone bill from your checking account.

Here's how it works:
1. Transfer $300 from your checking account to your savings account.
2. Use the remaining $200 in your checking account to cover the $250 phone bill.

Remember to check if there are any transaction limits or restrictions on transferring funds between accounts at NovaBank.



=== [B6] (out_of_scope) -- 2.9s ===
Q: Can you give me a recipe for chocolate chip cookies?
A (base SLM): I'm sorry, but I can't assist with that. My capabilities are limited to providing information related to NovaBank's banking services such as cards, loans, mortgages, accounts, transfers, ATMs, passwords, and fees. I don't have the ability to provide recipes or any other non-financial content.



=== [B7] (in_domain_factual) -- 7.1s ===
Q: What is the difference between blocking a credit card and cancelling it?
A (base SLM): Blocking a credit card involves temporarily preventing transactions from being processed on your account while keeping the card itself active for future use. This action allows you to manage your spending more effectively by stopping unauthorized charges or fraudulent activities.

Cancelling a credit card means permanently ending all access to that particular card. Once cancelled, the card will no longer be usable for any transactions, including those made in person at an ATM or online through our website or mobile app. Cancelling a card also removes its associated security features like PIN protection, which may need to be reactivated if you decide to reopen the card later.

Both actions have different implications depending on how they affect your financial situation and personal preferences regarding card usage. Blocking helps maintain control over your 

=== [B8] (in_domain_factual) -- 12.6s ===
Q: I want to dispute an ATM withdrawal that I did not make. Walk me through the process.
A (base SLM): Certainly! To dispute an ATM withdrawal that you didn't make, follow these steps:

1. **Review Your Statement**: First, check your recent bank statement for any transactions that might have been made at an ATM or another location where withdrawals were processed.

2. **Contact Your Bank**: Reach out to your bank's customer service department. You can usually do this by phone, email, or online chat. Provide them with details of the transaction, including the date, amount, and location.

3. **File a Dispute**: Once you've identified the incorrect transaction, file a formal dispute with your bank. This will typically involve providing documentation such as your account information, the disputed transaction details, and possibly a copy of your identification.

4. **Follow Up**: After submitting your dispute, keep track of any communications from y

## Free SLM memory before loading the (larger) reference LLM

In [5]:
del slm_model
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("SLM unloaded.")


SLM unloaded.


## System (B): Reference LLM -- No Fine-Tuning

`Qwen/Qwen2.5-7B-Instruct`, loaded with 4-bit NF4 quantisation (bitsandbytes) for **inference only** --
it is never fine-tuned, per PS2's rules for the reference model. Same family as the SLM (isolates the
effect of scale) and ~4.7x more parameters, satisfying PS2's "substantially larger, 7B+" requirement.

If your Virtual Lab GPU cannot fit even the 4-bit 7B model, see `README.md` -- "Reference LLM
alternatives" for a smaller (but still 7B+ where possible) or API-based fallback; do not silently swap
to a <7B model without updating the report's model-choice justification.

In [6]:
ref_model, ref_tokenizer = load_causal_lm(REFERENCE_LLM_MODEL_ID, quantize_4bit=True)
print(f"Loaded {REFERENCE_LLM_MODEL_ID} (4-bit)")
print(f"Parameters: {sum(p.numel() for p in ref_model.parameters()):,}")


W0811 23:41:16.025000 26200 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loaded Qwen/Qwen2.5-7B-Instruct (4-bit)
Parameters: 4,352,972,288


In [7]:
responses_B = {}
for item in benchmark:
    prompt_text = format_chat_prompt(ref_tokenizer, item["prompt"], SYSTEM_PROMPT)
    t0 = time.time()
    response = generate_response(ref_model, ref_tokenizer, prompt_text, **GENERATION_KWARGS)
    dt = time.time() - t0
    responses_B[item["id"]] = response
    print(f"=== [{item['id']}] ({item['type']}) -- {dt:.1f}s ===")
    print("Q:", item["prompt"])
    print("A (reference LLM):", response)
    print()


=== [B1] (in_domain_factual) -- 13.7s ===
Q: How do I activate a new debit card I just received from NovaBank?
A (reference LLM): To activate your new NovaBank debit card, follow these steps:

1. **Call Customer Service**: Dial 800-526-3226 to speak with a representative who can guide you through the activation process over the phone.

2. **Visit a Branch**: You can also visit any NovaBank branch where a teller will assist you in activating your card.

3. **Online Activation**: Log into your online banking account at novabank.com. Look for the "Card Services" or "Debit Card Services" section and follow the instructions provided there to activate your card.

4. **Mobile App**: If you have downloaded the NovaBank mobile app, open it and look for options related to card management or services. There should be an option to activate your new card within the app.

Once activated, you can start using your card for transactions. Remember to keep your PIN confidential and change it if you haven

=== [B2] (in_domain_factual) -- 12.4s ===
Q: What documents do I need to apply for a home loan with NovaBank?
A (reference LLM): To apply for a home loan with NovaBank, you will typically need to provide the following documents:

1. **Proof of Income**: W-2 forms, pay stubs, or tax returns from the past two years.
2. **Identification**: A government-issued ID such as a driver's license or passport.
3. **Employment Verification**: Letter from your current employer or recent employment history.
4. **Credit Report Authorization**: Consent to allow NovaBank to pull your credit report.
5. **Bank Statements**: Recent bank statements showing your financial activity.
6. **Down Payment Savings**: Documentation of funds available for the down payment and closing costs.
7. **Property Information**: Proof of property ownership if applicable, or details about the property you plan to purchase.

It’s best to check directly with NovaBank for their specific requirements, as they may have additional or

=== [B3] (in_domain_factual) -- 10.3s ===
Q: An ATM swallowed my card. What should I do?
A (reference LLM): If an ATM has swallowed your card, here’s what you should do:

1. **Stay Calm**: Do not panic.
2. **Check for the Card**: Look around the ATM area to see if your card is nearby or on the ground.
3. **Contact NovaBank**: Call our customer service line at [insert phone number] to report the issue. They will guide you through the next steps.
4. **Follow Up**: Keep track of any communication from the bank regarding your card retrieval.
5. **Secure Your Account**: Consider changing your PIN or updating your account details as a precaution.

We apologize for any inconvenience this may have caused. Please let us know if you need further assistance!



=== [B4] (formatting_sensitive) -- 3.5s ===
Q: Explain how to reset my online banking password. Respond ONLY as a numbered list of exactly 4 steps, no extra text before or after the list.
A (reference LLM): 1. Log in to your NovaBank Online Banking account.
2. Click on "Settings" or "Profile" in the menu.
3. Select "Change Password" or similar option.
4. Follow the prompts to set a new password.



=== [B5] (reasoning) -- 21.1s ===
Q: I have $500 in my checking account. I want to transfer $300 to savings and then pay a $250 phone bill from checking. Do I have enough balance for both, and in what order should I do them?
A (reference LLM): To ensure you have enough balance for both transactions, let's break it down:

1. **Transfer $300 from Checking to Savings:**
   - You currently have $500 in your checking account.
   - After transferring $300 to savings, your checking account will have:
     \[
     \$500 - \$300 = \$200
     \]

2. **Pay $250 Phone Bill from Checking:**
   - After the transfer, you have $200 left in your checking account.
   - Paying a $250 phone bill would require:
     \[
     \$200 - \$250 = -\$50
     \]
   - This results in an overdraft of $50.

**Conclusion:**
- Yes, you have enough balance to perform the first transaction (transfer $300 to savings), but not the second one (pay $250 phone bill) because after the transfer, you'll be short by $50.

**Order 

=== [B6] (out_of_scope) -- 3.2s ===
Q: Can you give me a recipe for chocolate chip cookies?
A (reference LLM): I'm sorry, but I can only provide information related to NovaBank's banking services. For a recipe for chocolate chip cookies, you might want to check out cooking websites or food blogs that offer recipes. Enjoy baking!



=== [B7] (in_domain_factual) -- 14.6s ===
Q: What is the difference between blocking a credit card and cancelling it?
A (reference LLM): At NovaBank, blocking a credit card and canceling a credit card serve different purposes:

1. **Blocking a Credit Card**: When you block a credit card, it temporarily suspends its use until you unblock it. This means that no transactions can be processed through the blocked card during this period. You might choose to block your card if you suspect fraudulent activity or if you need to take a break from using the card for personal reasons.

2. **Canceling a Credit Card**: Canceling a credit card permanently ends its functionality and removes it from your account. Once canceled, the card cannot be used for any transactions, and the account associated with the card will close. Cancellation typically involves returning the physical card and updating your records to reflect the change in status.

If you need to either block or cancel your credit card, ple

=== [B8] (in_domain_factual) -- 19.3s ===
Q: I want to dispute an ATM withdrawal that I did not make. Walk me through the process.
A (reference LLM): Certainly! Here’s how you can dispute an unauthorized ATM withdrawal from NovaBank:

1. **Contact Us Immediately**: Call our Customer Service line at [insert phone number] as soon as possible to report the unauthorized transaction.

2. **Provide Details**: When you call, be prepared to provide details such as:
   - The date of the transaction.
   - The amount withdrawn.
   - Any account numbers or card numbers related to the transaction.
   - A brief description of why you believe it was unauthorized.

3. **Follow Up in Writing**: After your initial call, follow up by sending a written statement to us via certified mail or email. Include all relevant information and any supporting documents like receipts or statements.

4. **Review Your Account**: Keep an eye on your account for any further transactions or communications regarding this is

## Save Verbatim Responses (required deliverable for Task 2)

In [8]:
baseline_results = {
    "benchmark": benchmark,
    "system_A_base_slm": {"model_id": SLM_MODEL_ID, "responses": responses_A},
    "system_B_reference_llm": {"model_id": REFERENCE_LLM_MODEL_ID, "responses": responses_B},
    "generation_kwargs": GENERATION_KWARGS,
}
save_json(baseline_results, os.path.join(RESULTS_DIR, "task2_baseline_responses.json"))
print("Saved results/task2_baseline_responses.json")

import pandas as pd
rows = []
for item in benchmark:
    rows.append({
        "id": item["id"], "type": item["type"], "prompt": item["prompt"],
        "A_base_slm": responses_A[item["id"]],
        "B_reference_llm": responses_B[item["id"]],
    })
side_by_side = pd.DataFrame(rows)
side_by_side.to_csv(os.path.join(RESULTS_DIR, "task2_side_by_side.csv"), index=False)
side_by_side


Saved results/task2_baseline_responses.json


,id,type,prompt,A_base_slm,B_reference_llm
0,B1,in_domain_factual,How do I activate a new debit card I just rece...,"To activate your new debit card at NovaBank, f...","To activate your new NovaBank debit card, foll..."
1,B2,in_domain_factual,What documents do I need to apply for a home l...,"To apply for a home loan with NovaBank, you ty...","To apply for a home loan with NovaBank, you wi..."
2,B3,in_domain_factual,An ATM swallowed my card. What should I do?,"If your ATM swallowed your card, it's importan...","If an ATM has swallowed your card, here’s what..."
3,B4,formatting_sensitive,Explain how to reset my online banking passwor...,1. Log in to your NovaBank account.\n2. Click ...,1. Log in to your NovaBank Online Banking acco...
4,B5,reasoning,I have $500 in my checking account. I want to ...,"Yes, you have enough balance for both transact...",To ensure you have enough balance for both tra...
5,B6,out_of_scope,Can you give me a recipe for chocolate chip co...,"I'm sorry, but I can't assist with that. My ca...","I'm sorry, but I can only provide information ..."
6,B7,in_domain_factual,What is the difference between blocking a cred...,Blocking a credit card involves temporarily pr...,"At NovaBank, blocking a credit card and cancel..."
7,B8,in_domain_factual,I want to dispute an ATM withdrawal that I did...,Certainly! To dispute an ATM withdrawal that y...,Certainly! Here’s how you can dispute an unaut...


## Limitations / Gap Table

Filled in from the actual outputs above (`results/task2_baseline_responses.json`, run locally on an
RTX 4050 Laptop GPU -- see report Appendix C for why this ran outside the Virtual Lab).

| Dimension | Where SLM (A) failed vs. LLM (B) | Evidence (prompt id) |
|---|---|---|
| Domain knowledge | A gives generic, non-NovaBank-branded procedures with no concrete channel details (no phone number, no branch/app names); B consistently grounds answers in NovaBank-specific channels (a phone number, novabank.com, "our" branding) | B1, B3 |
| Instruction / format adherence | No gap at baseline: A already produced exactly 4 numbered steps with no extra text for the strict-format prompt | B4 (both correct) |
| Completeness / reasoning | A failed the arithmetic reasoning task outright: it states "you have enough balance for both" and then describes paying a $250 bill from a $200 remaining balance -- an internal contradiction -- and never flags the resulting $50 overdraft. B performs the subtraction explicitly ($500-$300=$200, then $200-$250=-$50) and correctly concludes there isn't enough balance for both | B5 |
| Hallucination | A states a specific "60 days" dispute-resolution window with confident, unhedged phrasing but no NovaBank source backs that number; B avoids committing to an unverified specific timeframe | B8 |
| Persona / brand voice consistency | A rarely speaks as NovaBank itself ("your bank", "the ATM manufacturer") despite the system prompt instructing it to be the NovaBank assistant; B consistently uses "our", "NovaBank", and first-person service-desk phrasing | B1, B3, B7 |
| Out-of-scope handling | No gap at baseline: both A and B correctly declined the off-topic cookie-recipe request | B6 (both correct) |

**Inference:** The baseline gap is smaller than expected on refusal and strict-format compliance (B4,
B6) -- the 1.5B model already handles those. It is concentrated instead where it matters most: A's
arithmetic/logical reasoning (B5) isn't just weaker, it's outright wrong and self-contradictory, and A
consistently fails to sound like a NovaBank agent rather than a generic assistant. This makes
reasoning correctness and consistent NovaBank persona/branding the clearest fine-tuning targets --
format-following and scope-refusal are close to solved already and don't need much training signal.

## Conclusion

- Ran the frozen 8-prompt benchmark against (A) the unmodified `Qwen/Qwen2.5-1.5B-Instruct` and (B) the
  unmodified `Qwen/Qwen2.5-7B-Instruct` (4-bit, inference-only), using identical generation settings.
- Saved verbatim responses to `results/task2_baseline_responses.json` and
  `results/task2_side_by_side.csv` -- required inputs for Notebook 04 (diagnosis) and Notebook 05
  (three-way comparison).
- Completed the qualitative limitations/gap table above from the real outputs.

**Next notebook:** `03_finetuning_v1.ipynb` -- LoRA fine-tunes the SLM on the Notebook-01 training
split to produce Version 1 (v1), with the mandatory efficiency evidence (trainable-parameter % and
peak training memory).